Thoughts:
- llama and command r can decently simplify sentences zero-shot.
- add HSK vocabulary context so simplification includes more HSK1-4 vocab.
- add user-specific vocab knowledge which gets added to context. This will be complex words that the user knows (don't simplify) or complex words the user doesn't know (if it is a key word to the meaning, keep it, otherwise, simplify it)
- optional end-goals/simplification level: (1) extensive reading, keep it as simple as possible (2) a little intensive so the user can learn more words

Notebook for experimenting with bedrock LLM for text simplification

In [22]:
%load_ext autoreload
%autoreload 2
import boto3
from botocore.exceptions import ClientError
from langchain_aws.llms.bedrock import BedrockLLM
from langchain_aws.chat_models import ChatBedrockConverse
import utils.bedrock_pipeline as bp
import re
from bs4 import BeautifulSoup

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Create an Amazon Bedrock Runtime client.
brt = boto3.client("bedrock-runtime")

# Set the model ID
arn_cr = "cohere.command-r-v1:0"
arn_ds = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.deepseek.r1-v1:0"
arn_llama = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-2-3b-instruct-v1:0"
arn_llama2 = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-3-70b-instruct-v1:0"

llm_llama = ChatBedrockConverse(client=brt,
                        model_id=arn_llama,
                        provider="meta",
                        temperature=0.1,
                        max_tokens=150
                        ,)

llm_llama2 = ChatBedrockConverse(client=brt,
                        model_id=arn_llama2,
                        provider="meta",
                        temperature=0.1,
                        max_tokens=5000
                        ,)

llm_cr = ChatBedrockConverse(client=brt,
                        model=arn_cr,
                        temperature=0.1,
                        max_tokens=50,)

In [3]:
llm = llm_llama2

In [ ]:
import requests
import re
from bs4 import BeautifulSoup
url = 'https://read.douban.com/reader/column/69884011/chapter/622526435'
site = requests.get(url)
site.encoding = 'utf-8'
site_soup = BeautifulSoup(site.text, 'html.parser')

In [4]:
url = 'https://read.douban.com/reader/column/69884011/chapter/622526435'
soup = bp.extract_from_url(url)

In [ ]:
### For extracting sentences from <p> tags, while preserving quotations
sentences = []
for element in soup.find_all('p'):
    # print("Next:", element.decode_contents())
    paragraph = element.get_text(separator='', strip=True)  # Extract plain text without tags
    parts = re.split(r'([。！？!?．][”’」』"]?)', paragraph)
    
    # Recombine the split pieces into full sentences
    for i in range(0, len(parts) - 1, 2):
        sentence = parts[i] + parts[i + 1]
        sentence = sentence.strip()
        if sentence:
            sentences.append(sentence)

    # Handle any trailing part without punctuation
    if len(parts) % 2 == 1 and parts[-1].strip():
        sentences.append(parts[-1].strip())

In [7]:
### For extracting all text from <p> tags (multiple sentences)
sentences = []
for element in soup.find_all('p'):
    # print("Next:", element.decode_contents())
    paragraph = element.get_text(separator='', strip=True)  # Extract plain text without tags
    text = re.split(r'(?<=[。！？!?．])\s*', paragraph)
    sentences.append(paragraph)

In [8]:
len(sentences), sentences[:5]

(166,
 ['2021年初冬，李晟最后一次见到孟北晗。',
  '凌晨五点到达云南丽江，走在古老的街道上，身边空无一人。寒风凛冽，天寒地冻，远山高大寂寥的轮廓在茫茫晨雾中依稀可见。李晟走了很久，才看到孟北晗的身影从雾气尽头慢慢浮现。',
  '孟北晗那头栗色长卷发没来得及打理，散乱地披在身后，抱歉地说：“小乐生病了，我刚从医院赶过来。”',
  '李晟立刻紧张起来：“小乐怎么了？严不严重？”',
  '孟北晗摇摇头，目光柔和：“没事。可能是昨晚吃坏了肚子，凌晨时候吐了。医生说是急性胃肠炎，不碍事。”'])

In [10]:
simplification_prompt = bp.build_simplification_prompt(sentences)

In [11]:
simplified_raw = llm.invoke(simplification_prompt).content
simplified = bp.parse_llm_output(simplified_raw)

In [12]:
judge_prompt = bp.build_judge_prompt(sentences, simplified)

In [13]:
judged_raw = llm.invoke(judge_prompt).content
judged = bp.parse_llm_output(judged_raw)

In [ ]:
ner_prompt = bp.build_NER_prompt(judged)

In [39]:
ner_raw = llm.invoke(ner_prompt).content
ner = bp.parse_llm_output(ner_raw)

In [40]:
import webbrowser
html = bp.generate_html(url, ner)
new_soup = BeautifulSoup(html, features="html.parser") # parse to html

with open("output.html", "w", encoding="utf-8") as file:
    file.write(str(new_soup)) # write html to file
webbrowser.open("output.html");

In [17]:
bp.corpus_metrics(sentences[:len(judged)], judged)

(np.float64(-0.8429261020124641), np.float64(0.2587944686426238))